# Módulo AI3 · RAG y agentes — Práctica guiada

**Producto:** *asistente de soporte de TiendaSur* con conocimiento propio.

1. **RAG:** chunking de la guía de pedidos → "embeddings" (TF-IDF como stand-in) → top-k → contexto citado
2. **Generación:** respuesta con citas (con API de Claude si tienes clave; mock si no)
3. **Agente:** herramientas `buscar_docs` + `get_pedido` + `crear_ticket` y bucle de decisión

> Todo corre en CPU. La celda [API] opcional mejora la generación final con Claude.


In [ ]:
import os
import json
import re
import textwrap
import numpy as np
import pandas as pd
from collections import Counter

try:
    import google.colab  # noqa
    from google.colab import userdata
    os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY") or "")
except ImportError:
    pass

TIENE_CLAVE = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("clave de API presente:", TIENE_CLAVE)

## 3.1 · Documento propio: la guía de pedidos

In [ ]:
GUIA = """## Guía de pedidos de TiendaSur

### Envíos
El envío estándar tarda 3-5 días laborables. Los pedidos urgentes
llegan al día siguiente si se hacen antes de las 14:00.

### Facturas
La factura se emite el mismo día de la compra y se envía por correo.
Si necesitas factura con CIF, se emite en 24 h con datos fiscales completos.

### Devoluciones
Hay 30 días naturales para devolver. El producto debe estar sin usar.
El envío de devolución corre a cargo del cliente salvo producto defectuoso.

### Problemas de entrega
Si el paquete llega dañado, haz un ticket con foto en menos de 48 h.
Si no llega en 7 días, contacta con soporte: el pedido se re-envía gratis."""

# --- 3.3 · Chunking por secciones (con título como prefijo) ----------------
def chunking_secciones(doc: str):
    """Trocea por secciones (###) y añade el título como prefijo del chunk."""
    # quitamos el encabezado principal ("## ...") y troceamos por secciones
    cuerpo = doc.split("\n\n", 1)[1]
    partes = ["### " + p for p in re.split(r"\n###\s+", cuerpo) if p.strip()]
    chunks = []
    for p in partes:
        lineas = p.strip().split("\n")
        titulo = lineas[0].replace("### ", "").strip()
        cuerpo_sec = "\n".join(lineas[1:]).strip()
        chunks.append({"id": titulo, "texto": f"{titulo}: {cuerpo_sec}"})
    return chunks

chunks = chunking_secciones(GUIA)
print(f"{len(chunks)} chunks")
for c in chunks:
    print(f"  - [{c['id']}] {c['texto'][:70]}…")

**Lectura:** troceamos por sección (estructura del documento), con el título como prefijo dentro del texto — así el vector "contiene" el tema y citar es trivial (`[Envíos]`). El solape aquí no es necesario porque las secciones son cortas y completas; en documentos largos lo añade el ejercicio 2.


## 3.2 + 3.4 · Embeddings (stand-in TF-IDF) y búsqueda

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Stand-in de embeddings: TF-IDF (bag of words). Con embeddings reales (sbert/OpenAI)
# solo cambia esta función `encodar`; el resto del pipeline es idéntico.
_vec = TfidfVectorizer(ngram_range=(1, 2))
_mat_chunks = _vec.fit_transform([c["texto"] for c in chunks]).toarray()

def encodar(texto: str) -> np.ndarray:
    return _vec.transform([texto]).toarray()[0]

def buscar(top_k: int = 2, **ctx):
    """Recuperación: coseno entre la consulta y todos los chunks."""
    consulta = ctx["consulta"]
    q = encodar(consulta)
    sims = (_mat_chunks @ q) / (np.linalg.norm(_mat_chunks, axis=1) * np.linalg.norm(q) + 1e-9)
    idx = np.argsort(sims)[::-1][:top_k]
    return [{"id": chunks[i]["id"], "texto": chunks[i]["texto"], "score": float(sims[i])} for i in idx]

for r in buscar(consulta="cuánto tarda el envío"):
    print(f"[{r['id']}] score={r['score']:.2f} → {r['texto'][:60]}…")

## 3.5 · Generación con contexto citado

In [ ]:
# System prompt de RAG: reglas que evitan el "hallucination de cortesía"
SYSTEM_RAG = textwrap.dedent("""
Eres el asistente de pedidos de TiendaSur.

Responde SOLO con la información de los fragmentos entre <docs>.
- Cita el fragmento con [id del chunk].
- Si el fragmento no contesta la pregunta, di exactamente: "no lo sé".
""")
print(SYSTEM_RAG)

In [ ]:
# [API] generación con Claude; sin clave: respuesta "mock" con los chunks recuperados
def responder_rag(consulta: str, top_k: int = 2):
    docs = buscar(top_k=top_k, consulta=consulta)
    contexto = "\n\n".join(f'<docs id="{d["id"]}">{d["texto"]}</docs>' for d in docs)
    prompt = f"""<docs>
{contexto}
</docs>

Consulta: {consulta}"""
    if TIENE_CLAVE:
        import anthropic
        msg = anthropic.Anthropic().messages.create(
            model="claude-opus-5-5", system=SYSTEM_RAG, max_tokens=4000,
            output_config={"effort": "low"},
            messages=[{"role": "user", "content": prompt}],)
        respuesta = next(b.text for b in msg.content if b.type == "text")   # puede haber un bloque thinking antes
    else:
        respuesta = "\n".join(f"- {d['texto']}" for d in docs) + "\n[SALTADO: sin clave — respuesta mock con los chunks]"
    print(f"Q: {consulta}\nDocs: {[d['id'] for d in docs]}\nR: {respuesta}\n")

for q in ["¿cuánto tarda el envío?", "puedo devolver algo usado?", "¿cuál es la capital de Francia?"]:
    responder_rag(q)

**Lectura:** la tercera pregunta debe terminar en "no lo sé" — el system prompt lo permite y evita que el modelo improvisa. Con TF-IDF la recuperación por keywords funciona sorprendentemente bien; con embeddings semánticos (sbert) mejora para sinónimos ("reenviar" vs "envío de nuevo"): la prueba la haces en el ejercicio 3.


## 3.6 · Agente: tools + bucle de decisión

In [ ]:
TICKETS = []
from typing import Optional

def crear_ticket(tema: str, pedido_id: Optional[int] = None) -> dict:
    """Crea un ticket de soporte."""
    t = {"id": len(TICKETS) + 1, "tema": tema, "pedido": pedido_id}
    TICKETS.append(t)
    return t

def get_pedido(pedido_id: int) -> dict:
    PEDIDOS = {1042: {"producto": "Tazas TiendaSur", "estado": "preparando", "fecha": "2026-02-06"}}
    return PEDIDOS.get(pedido_id, {"error": f"pedido {pedido_id} no existe"})

TOOLS = {
    "buscar_docs":   {"desc": "Busca en la guía de pedidos de TiendaSur y devuelve documentos relevantes.",
                      "fn": buscar},
    "get_pedido":    {"desc": "Estado de un pedido por id.",
                      "fn": lambda pedido_id: get_pedido(pedido_id)},
    "crear_ticket":  {"desc": "Crea un ticket de soporte por tema; opcionalmente con id de pedido.",
                      "fn": crear_ticket},
}

AGENTE_SYSTEM = textwrap.dedent(SYSTEM_RAG + """

Además tienes herramientas útiles para resolver incidencias:
- `buscar_docs(consulta)`: busca en la guía.
- `get_pedido(pedido_id)`: estado de un pedido.
- `crear_ticket(tema, pedido_id)`: registra la incidencia y devuelve su id.

Plan: consulta las docs o el pedido que necesites, y para problemas reales
crea el ticket y ciérrate respondiendo el id.""")

In [ ]:
TOOL_SCHEMAS = [
    {"name": "buscar_docs", "description": TOOLS["buscar_docs"]["desc"],
     "input_schema": {"type": "object", "properties": {"consulta": {"type": "string"}},
                       "required": ["consulta"]}},
    {"name": "get_pedido", "description": TOOLS["get_pedido"]["desc"],
     "input_schema": {"type": "object", "properties": {"pedido_id": {"type": "integer"}},
                       "required": ["pedido_id"]}},
    {"name": "crear_ticket", "description": TOOLS["crear_ticket"]["desc"],
     "input_schema": {"type": "object", "properties": {"tema": {"type": "string"},
                                                        "pedido_id": {"type": "integer"}},
                       "required": ["tema"]}},
]

def ejecutar(nombre: str, args: dict):
    return TOOLS[nombre]["fn"](**args)

In [ ]:
def agente(tarea: str, max_vueltas: int = 5):
    """[API] Bucle del agente: decide → usa tools → responde. Sin API: mock del plan."""
    if not TIENE_CLAVE:
        print("[SALTADA: sin clave — flujo mock del plan del agente]")
        docs = buscar(consulta="paquete dañado")
        print("1) el agente consultaría buscar_docs('paquete dañado') →", [d["id"] for d in docs])
        print("2) crear_ticket(tema='paquete dañado') →", crear_ticket("paquete dañado (mock)"))
        return
    import anthropic
    c = anthropic.Anthropic()
    mensajes = [{"role": "user", "content": tarea}]
    for vuelta in range(max_vueltas):
        resp = c.messages.create(model="claude-opus-5-5", system=AGENTE_SYSTEM,
                                  max_tokens=8000, tools=TOOL_SCHEMAS, messages=mensajes)
        if resp.stop_reason != "tool_use":
            final = next((b.text for b in resp.content if b.type == "text"), "")
            print("→ respuesta final:", final[:300])
            return
        resultados = []   # una tool_result por CADA tool_use (llamadas paralelas), en un solo mensaje
        for bloque in (b for b in resp.content if b.type == "tool_use"):
            print(f"vuelta {vuelta+1}: {bloque.name}({bloque.input})")
            resultado = ejecutar(bloque.name, bloque.input)
            print("   resultado:", str(resultado)[:150])
            resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                               "content": json.dumps(resultado)})
        mensajes.append({"role": "assistant", "content": resp.content})
        mensajes.append({"role": "user", "content": resultados})
    print("(límite de vueltas: el agente no terminó — revisa la herramienta o el prompt)")

print("— Incidencia —")
agente("Mi pedido 1042 llegó con una taza rota. ¿Se puede devolver? Registra el problema")

## 3.7 · Cuándo NO agente (el anti-teatro)

In [ ]:
def soporte_router(consulta: str) -> str:
    """Router determinista: la mitad de los 'agentes' necesitan solo esto."""
    docs = buscar(top_k=1, consulta=consulta)
    tema = docs[0]["id"] if docs else "otros"
    if tema == "Problemas de entrega":
        t = crear_ticket(tema=f"incidencia: {consulta}")
        return f"He abierto el ticket #{t['id']}. Un humano te responderá."
    return docs[0]["texto"]

print(soporte_router("el paquete llegó en dos trozos"))
print(soporte_router("¿cuánto tarda el envío?"))

**Lectura:** este router resuelve el 80 % de los casos con **0 llamadas a herramientas del modelo**. El agente del bucle anterior se justifica cuando la secuencia de acciones no se puede anticipar — y si se puede, mejor un script.


---

## Resumen del notebook

- **RAG**: chunking con estructura → vectorize → top-k → prompt con docs y permiso para "no lo sé".
- **TF-IDF** demuestra el pipeline; cambiar `encodar` por sbert/OpenAI lo lleva a producción.
- **Agente**: el AI2 (tools) + planificación y límites. Router determinista para el 80 %.

---

## Ejercicios (5) — entregables

**1.** Añade solape al chunking: haz 2 chunks por sección (mitad inicial + mitad final, con solape del 15 %). ¿Mejora alguna respuesta? ¿Qué pregunta se rota?

**2.** Carga la guía con **2 documentos más** (uno de devoluciones extendido). ¿El router sigue funcionando? ¿Qué pregunta empieza a confundir secciones?

**3.** *(Opcional, Colab o local con HF)* Sustituye TF-IDF por embeddings de `sentence-transformers` (model `paraphrase-multilingual-MiniLM-L12-v2`). Repite la evaluación del caso práctico: ¿cuántas de tus 5 preguntas pasan de fallar a acertar?

**4.** En el agente, añade una herramienta `responder(tema)` que el sistema use para "cerrar" la tarea en vez de la respuesta final. ¿Qué problema introduces? (pista: bucles y cómo termina).

**5.** Reescribe el caso "quiero cambiar la dirección de un pedido" como **router determinista** (2 ifs y 2 respuestas). Escribe en 3 líneas por qué es mejor que un agente aquí.
